In [1]:
import os
os.listdir('/kaggle/input/datasets/samuelcortinhas/muffin-vs-chihuahua-image-classification')

['test', 'train']

In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = '/kaggle/input/datasets/samuelcortinhas/muffin-vs-chihuahua-image-classification/train'
test_dir  = '/kaggle/input/datasets/samuelcortinhas/muffin-vs-chihuahua-image-classification/test'

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir, target_size=(224,224),
    batch_size=32, class_mode='binary', subset='training'
)
val_data = train_datagen.flow_from_directory(
    train_dir, target_size=(224,224),
    batch_size=32, class_mode='binary', subset='validation'
)
test_data = test_datagen.flow_from_directory(
    test_dir, target_size=(224,224),
    batch_size=32, class_mode='binary', shuffle=False
)

2026-03-12 15:14:51.555147: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773328491.791901      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773328491.861935      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773328492.395723      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773328492.395778      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773328492.395781      23 computation_placer.cc:177] computation placer alr

Found 3788 images belonging to 2 classes.
Found 945 images belonging to 2 classes.
Found 1184 images belonging to 2 classes.


In [3]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False

model = Sequential([
    base_model,
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)

model.fit(train_data, validation_data=val_data, epochs=10, callbacks=[early_stop])

I0000 00:00:1773328523.832536      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1773328534.454726      85 service.cc:152] XLA service 0x7f73d8010cc0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773328534.454778      85 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1773328536.082490      85 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1773328543.268049      85 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


119/119 ━━━━━━━━━━━━━━━━━━━━ 128s 948ms/step - accuracy: 0.9409 - loss: 0.6285 - val_accuracy: 0.9820 - val_loss: 0.3179
Epoch 2/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 79s 664ms/step - accuracy: 0.9894 - loss: 0.1533 - val_accuracy: 0.9841 - val_loss: 0.1632
Epoch 3/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 78s 658ms/step - accuracy: 0.9896 - loss: 0.0949 - val_accuracy: 0.9884 - val_loss: 0.1011
Epoch 4/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 77s 645ms/step - accuracy: 0.9877 - loss: 0.0582 - val_accuracy: 0.9852 - val_loss: 0.1015
Epoch 5/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 77s 651ms/step - accuracy: 0.9934 - loss: 0.0366 - val_accuracy: 0.9862 - val_loss: 0.1528
Epoch 6/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 77s 652ms/step - accuracy: 0.9942 - loss: 0.0308 - val_accuracy: 0.9810 - val_loss: 0.2387


In [4]:
loss, acc = model.evaluate(test_data)
print(f"Test Accuracy: {acc:.4f}")

37/37 ━━━━━━━━━━━━━━━━━━━━ 12s 337ms/step - accuracy: 0.9967 - loss: 0.0369
Test Accuracy: 0.9907


In [5]:
import numpy as np
import pandas as pd

pred = model.predict(test_data)

# ถ้าค่า > 0.5 ให้เป็น 1, น้อยกว่านั้นเป็น 0
labels = (pred > 0.5).astype(int).flatten()


# บางทีระบบต้องการแค่ชื่อไฟล์ ไม่เอา 'test/' นำหน้า
filenames = [f.split('/')[-1] for f in test_data.filenames]


submission = pd.DataFrame({
    'id': filenames,
    'label': labels
})

#  บันทึกไฟล์
submission.to_csv("submission.csv", index=False)
print("submission.csv created successfully!")

37/37 ━━━━━━━━━━━━━━━━━━━━ 12s 200ms/step
submission.csv created successfully!
